In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (SparkSession.builder.appName("DataLakeDemo").master("local[*]").getOrCreate())
print("Spark started successfully!")

Spark started successfully!


In [2]:
RAW_CUSTOMERS = "data_lake/raw/customers"
RAW_ORDERS = "data_lake/raw/orders"
PROCESSED_CUSTOMERS = "data_lake/processed/customers"
PROCESSED_ORDERS = "data_lake/processed/orders"
CURATED_SALES = "data_lake/curated/sales"

In [3]:
customers_df = (spark.read.option("header", "true").option("inferSchema", "true").csv(RAW_CUSTOMERS))

customers_df.show()
customers_df.printSchema()

+-----------+------+---------+
|customer_id|  name|     city|
+-----------+------+---------+
|          1| Rahul|Bangalore|
|          2|Ananya|   Mysore|
|          3| Arjun|  Chennai|
|          4| Priya|Hyderabad|
+-----------+------+---------+

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)



In [4]:
orders_df = spark.read.json(RAW_ORDERS)

orders_df.show()
orders_df.printSchema()

+-----------+--------+-----+--------+--------+
|customer_id|order_id|price| product|quantity|
+-----------+--------+-----+--------+--------+
|          1|     101|50000|  Laptop|       2|
|          2|     102|  800|   Mouse|       3|
|          1|     103| 1500|Keyboard|       1|
|          3|     104|12000| Monitor|       2|
+-----------+--------+-----+--------+--------+

root
 |-- customer_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- price: long (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)



In [5]:
customers_processed = (customers_df.withColumn("name", col("name").cast("string")).withColumn("city", col("city").cast("string")))

orders_processed = (orders_df.withColumn("quantity", col("quantity").cast("integer")).withColumn("price", col("price").cast("double")))

In [6]:
customers_processed.write.mode("overwrite").parquet(PROCESSED_CUSTOMERS)

orders_processed.write.mode("overwrite").parquet(PROCESSED_ORDERS)

print("Processed data written successfully.")

Processed data written successfully.


In [7]:
sales_df = (
    orders_processed
    .join(
        customers_processed,
        orders_processed.customer_id ==
        customers_processed.customer_id,
        "left"
    )
)

sales_df.show()

+-----------+--------+-------+--------+--------+-----------+------+---------+
|customer_id|order_id|  price| product|quantity|customer_id|  name|     city|
+-----------+--------+-------+--------+--------+-----------+------+---------+
|          1|     101|50000.0|  Laptop|       2|          1| Rahul|Bangalore|
|          2|     102|  800.0|   Mouse|       3|          2|Ananya|   Mysore|
|          1|     103| 1500.0|Keyboard|       1|          1| Rahul|Bangalore|
|          3|     104|12000.0| Monitor|       2|          3| Arjun|  Chennai|
+-----------+--------+-------+--------+--------+-----------+------+---------+



In [8]:
sales_df = sales_df.withColumn("total_amount",col("quantity") * col("price"))

In [9]:
sales_df = sales_df.select(
    orders_processed.order_id,
    orders_processed.customer_id,
    customers_processed.name,
    customers_processed.city,
    orders_processed.product,
    orders_processed.quantity,
    orders_processed.price,
    col("total_amount")
)

sales_df.show()

+--------+-----------+------+---------+--------+--------+-------+------------+
|order_id|customer_id|  name|     city| product|quantity|  price|total_amount|
+--------+-----------+------+---------+--------+--------+-------+------------+
|     101|          1| Rahul|Bangalore|  Laptop|       2|50000.0|    100000.0|
|     102|          2|Ananya|   Mysore|   Mouse|       3|  800.0|      2400.0|
|     103|          1| Rahul|Bangalore|Keyboard|       1| 1500.0|      1500.0|
|     104|          3| Arjun|  Chennai| Monitor|       2|12000.0|     24000.0|
+--------+-----------+------+---------+--------+--------+-------+------------+



In [10]:
sales_df.write.mode("overwrite").parquet(CURATED_SALES)

print("Curated sales data written successfully.")

Curated sales data written successfully.
